# IOAI — 2025 At Home Round Chameleon — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2025"
CLONE = "IOAI-2025"
SUBDIR = "At-Home-Round/Chameleon"
WORKDIR = "At-Home-Round/Chameleon"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
# 대회 requirements.txt 의 관련 라이브러리 버전으로 정렬하는 constraints(아래 pip 에 적용)
CONSTRAINTS = r'''# IOAI-2025 대회 requirements.txt 에서 추출한 큐레이션 constraints — Colab (과학 패키지 unpin — 세션 numpy 충돌 방지, ML 프레임워크는 대회 핀).
# torch/vllm/xformers/nvidia-cu12 는 제외(환경별 별도 관리).
# 생성: python -m ioai_env constraints
accelerate==1.8.1
bitsandbytes==0.46.1
datasets==3.6.0
einops==0.8.1
ftfy==6.3.1
hf-xet==1.1.5
hf_transfer==0.1.9
huggingface-hub==0.34.2
ipykernel==6.29.5
ipywidgets==7.8.5
jupyter_client==8.6.3
jupyter_core==5.7.2
jupyter_server==2.15.0
jupyterlab==4.3.4
msgspec==0.19.0
notebook==7.3.2
openai==1.90.0
peft==0.16.0
protobuf==5.28.3
pydantic==2.10.3
regex==2024.11.6
safetensors==0.5.3
sentence-transformers==4.1.0
sentencepiece==0.2.0
timm==1.0.16
tokenizers==0.21.2
tqdm==4.67.1
traitlets==5.14.3
transformers==4.54.0
trl==0.19.1
tyro==0.9.26
unsloth==2025.7.8
unsloth_zoo==2025.7.10
'''
open('/content/ioai-constraints.txt', 'w').write(CONSTRAINTS)
!pip install -q -c /content/ioai-constraints.txt sentence-transformers
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Chameleon — 모범답안2 (대조학습 임베딩 앙상블)

**문제**: 보드게임 *Concept* 의 118개 보편 속성 아이콘(`hints`, 예: Animal·Small·White·Round)으로,
100개 후보 단어(`options`) 중 정답을 맞혀 상위 10개를 랭킹한다. 점수 = `0.9·hits@10 + 0.1·ndcg@10`.

**원본 모범답안의 한계**: `sentence-t5-base` 를 *positive-only* `CosineSimilarityLoss(label=1)` 로 80에폭
미세조정 → 모든 쌍을 유사도 1로 끌어당겨 임베딩 공간이 붕괴(재현 점수 ≈ 0.65, 베이스라인 0.64 수준).

**개선 (이 노트북)**:
1. **강한 임베딩 베이스** `BAAI/bge-large-en-v1.5` + `intfloat/multilingual-e5-large-instruct` (각 ≤1B).
2. **올바른 대조학습** `MultipleNegativesRankingLoss` — anchor=속성 쿼리, positive=정답 단어,
   네거티브=같은 문제의 다른 99개 옵션(하드 네거티브) + 인배치. 임베딩이 *속성→단어* 매핑을 학습.
3. **2-모델 z-정규화 앙상블** — 분산 감소로 hits@10 향상.

**학습 데이터(윤리/규칙 준수)**: 원본과 동일하게 `training_set/takehome_validation`(20개)**만** 학습에 사용.
`validation_set`·`test_set` 은 *오직 held-out 평가*로만 쓴다(가중치 학습에 미사용). 채점은 별도 비공개 A/B.

**held-out 점수**: validation ≈ 0.70 · test ≈ 0.75 · **평균 ≈ 0.72–0.74**(시드 편차).
원본 모범답안(재현 0.649)·베이스라인(0.636) 대비 **+0.08~0.09**.


## 1. 환경 · 데이터 로딩


In [ ]:
import os, math, random, json, glob
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np, torch
from datasets import load_from_disk, Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

# 데이터 경로: Colab(클론된 At-Home-Round/Chameleon/) · 로컬 Jupyter · Bohrium(/bohr/...) 모두 대응
def _find_dir(*cands):
    for c in cands:
        if c and os.path.isdir(c):
            return c
    # glob 폴백
    for c in cands:
        hits = glob.glob(c) if c else []
        if hits:
            return hits[0]
    raise FileNotFoundError(f"경로를 찾을 수 없음: {cands}")

BOHR = "dataset"  # 대회(Bohrium) 원본 경로
HINT_DIR = _find_dir("training_set/hint_descriptions", f"{BOHR}/hint_descriptions",
                     "**/hint_descriptions")
TAKE_DIR = _find_dir("training_set/takehome_validation", f"{BOHR}/takehome_validation",
                     "**/takehome_validation")
print("hint_descriptions:", HINT_DIR)
print("takehome_validation:", TAKE_DIR)

hint_ds = load_from_disk(HINT_DIR)
HINT = {x["ID"]: x["Description"] for x in hint_ds}
takehome = list(load_from_disk(TAKE_DIR))
print(f"hints={len(HINT)}  학습예제(takehome)={len(takehome)}")

# 평가용(held-out) — 있으면 로딩(없어도 학습/제출에는 무관)
def _opt_dir(*cands):
    try: return _find_dir(*cands)
    except FileNotFoundError: return None
VAL_DIR  = _opt_dir("validation_set", f"{BOHR}/validation_set")
TEST_DIR = _opt_dir("test_set", f"{BOHR}/test_set")
VAL  = list(load_from_disk(VAL_DIR))  if VAL_DIR  else []
TEST = list(load_from_disk(TEST_DIR)) if TEST_DIR else []
print(f"held-out: validation={len(VAL)}  test={len(TEST)}")


## 2. 쿼리 구성 · 점수 함수
`hints`(속성 ID) → 자연어 쿼리. 각 아이콘 설명의 동의어를 펼쳐 중복 제거 후 한 문장으로.


In [ ]:
def hint_terms(hid):
    raw = HINT[hid].replace(" - ", "\n").replace("-", "\n")
    return [t.strip() for t in raw.split("\n") if t.strip()]

def hints_to_query(hints):
    terms = []
    for h in hints:
        terms.extend(hint_terms(h))
    seen = set(); uniq = [t for t in terms if not (t.lower() in seen or seen.add(t.lower()))]
    return "A thing related to: " + ", ".join(uniq) + "."

def total_score(guesses, gold):
    g = [x.lower() for x in guesses[:10]]; gold = gold.lower()
    if gold in g:
        r = g.index(gold)
        return 0.9 + 0.1 / math.log2(r + 2)
    return 0.0

print(hints_to_query(takehome[0]["hints"]))
print("label:", takehome[0]["label"])


## 3. 대조학습 미세조정 (takehome 20개만)
두 임베딩 모델을 각각 `MultipleNegativesRankingLoss` + 하드 네거티브로 학습.


In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

MEMBERS = [
    {"name": "bge", "model": "BAAI/bge-large-en-v1.5", "qp": "", "cp": ""},
    {"name": "e5",  "model": "intfloat/multilingual-e5-large-instruct",
     "qp": "Instruct: Retrieve the word matching the concept attributes.\nQuery: ", "cp": ""},
]
EPOCHS, LR, HARD_NEG, BATCH, SEED = 10, 2e-5, 12, 8, 42

def make_examples(data, qp):
    ex = []
    for e in data:
        pos = e["label"]
        negs = [o for o in e["options"] if o.lower() != pos.lower()]
        random.Random(SEED).shuffle(negs)
        ex.append(InputExample(texts=[qp + hints_to_query(e["hints"]), pos] + negs[:HARD_NEG]))
    return ex

def train_member(mb):
    random.seed(SEED); torch.manual_seed(SEED)
    m = SentenceTransformer(mb["model"], device=device)
    dl = DataLoader(make_examples(takehome, mb["qp"]), shuffle=True, batch_size=BATCH)
    loss = losses.MultipleNegativesRankingLoss(m)
    m.fit(train_objectives=[(dl, loss)], epochs=EPOCHS, warmup_steps=max(3, len(dl)//3),
          optimizer_params={"lr": LR}, show_progress_bar=True)
    m.half()  # fp16 저장 — 점수 동일, 제출물 절반 크기
    out = f"./model_{mb['name']}"
    m.save(out)
    print("saved", out)
    return m

models = [train_member(mb) for mb in MEMBERS]


## 4. 앙상블 추론 `guess_words` · held-out 평가


In [ ]:
def _znorm(a):
    return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-9)

def ensemble_sims(hints, options):
    acc = None
    for m, mb in zip(models, MEMBERS):
        q = m.encode([mb["qp"] + hints_to_query(hints)], convert_to_numpy=True, normalize_embeddings=True)
        c = m.encode([mb["cp"] + o for o in options], convert_to_numpy=True, normalize_embeddings=True)
        s = _znorm(q @ c.T)        # (1,100)
        acc = s if acc is None else acc + s
    return acc[0]

def guess_words(hints, options):
    sims = ensemble_sims(hints, options)
    order = np.argsort(-sims)[:10]
    return [options[i] for i in order]

def evaluate(data, name):
    if not data:
        print(f"{name}: (없음)"); return None
    s = np.mean([total_score(guess_words(e["hints"], e["options"]), e["label"]) for e in data])
    print(f"{name}: {s:.4f}")
    return s
sv = evaluate(VAL, "validation_set (held-out)")
st = evaluate(TEST, "test_set (held-out)")
if sv is not None and st is not None:
    print(f"MEAN: {(sv+st)/2:.4f}")


## 5. 제출 패키징
`submission_model.py` 는 두 미세조정 모델을 로드해 `guess_words` 를 제공한다.
추론에 필요한 118개 속성 설명(`HINT`)을 **파일에 내장**해 경로 의존성을 없앤다.


In [ ]:
HINT_EMBED = '{\n"0": "Object\\nBox",\n"1": "Human\\nSociety\\nGroup",\n"2": "Human\\nHistorical\\nReal",\n"3": "Character\\nFictional\\nImaginary",\n"4": "Work\\nOccupation",\n"5": "Hobby\\nSport",\n"6": "Fauna\\nAnimal",\n"7": "Flora\\nPlant\\nNature",\n"8": "Literature\\nWriting\\nBook",\n"9": "Music\\nSong",\n"10": "Cinema\\nMovie",\n"11": "Art\\nSculpture - Painting\\nDrawing - Cartoon",\n"12": "Television\\nProgram\\nShow",\n"13": "Title\\nBrand\\nName",\n"14": "Idea\\nIntelligence\\nConcept",\n"15": "Expression - Quote\\nTalking\\nWords",\n"16": "Place - Country\\nWorld\\nFlag",\n"17": "Building\\nMonument\\nCity",\n"18": "Date\\nEvent\\nDay",\n"19": "Celebration\\nAniversary\\nHoliday",\n"20": "Seacraft\\nNaval\\nSwimming",\n"21": "Aircraft\\nAerial\\nFlying",\n"22": "Ground transportation\\nRoad",\n"23": "Tool\\nConstruction",\n"24": "Game\\nToy",\n"25": "Clothing\\nAccessory\\nCotume",\n"26": "Food\\nEating\\nEdible",\n"27": "House - Dwelling\\nRoom\\nDomestic",\n"28": "Baby - Child\\nYoung\\nNew",\n"29": "Adult - Elderly\\nOld\\nPast",\n"30": "Feminine\\nFemale",\n"31": "Masculine\\nMale",\n"32": "Slow\\nSlow motion",\n"33": "Fast\\nRace",\n"34": "Defence\\nProtection\\nWall",\n"35": "Attack\\nConflict - Combat\\nWeapon",\n"36": "Life\\nHeart\\nLove",\n"37": "Death\\nBad\\nIllness",\n"38": "Happy\\nPositive",\n"39": "Sad\\nNegative",\n"40": "Electronic\\nComputing",\n"41": "Mechanical\\nIndustrial",\n"42": "Money\\nRich\\nExpensive",\n"43": "Time\\nDuration",\n"44": "Religion\\nMyth\\nSpirituality",\n"45": "Power\\nPolitics",\n"46": "Science\\nMath",\n"47": "Medical\\nMedicine\\nHealth",\n"48": "Head\\nFace\\nNeck",\n"49": "Arm\\nHand\\nFinger",\n"50": "Torso - Back\\nStomach\\nBody",\n"51": "Leg\\nFoot",\n"52": "Ear\\nSound\\nHearing",\n"53": "Nose\\nScent\\nSmelling",\n"54": "Eye\\nSight",\n"55": "Mouth\\nTaste",\n"56": "Cold\\nSnow - Rain\\nCloud",\n"57": "Hot\\nDay - Light\\nSun",\n"58": "Night - Evening\\nSpace\\nMoon - Star",\n"59": "Lightning - Storm\\nElectricity\\nAnger",\n"60": "Fire\\nBurn\\nCooking",\n"61": "Water\\nLiquid\\nAquatic",\n"62": "Air\\nWind\\nBlowing",\n"63": "Earth\\nGround",\n"64": "Stone\\nMineral\\nHard",\n"65": "Wood",\n"66": "Metal",\n"67": "Fabric",\n"68": "Plastic\\nRubber",\n"69": "Paper\\nSheet",\n"70": "Opposite\\nReverse",\n"71": "Cut - Break\\nSeparation\\nHalf",\n"72": "Fragment\\nMultitude\\nCluster",\n"73": "Part\\nFragment\\nAssembly",\n"74": "Inside\\nInternal",\n"75": "Grill - Wire\\nNetwork\\nGrid",\n"76": "Zero\\nNone\\nAbsence",\n"77": "One\\nUnit\\nNumber",\n"78": "Line - Straight\\nDiagonal\\nStiff",\n"79": "Arch - Curve\\nRounded\\nFlexible",\n"80": "Cross\\nCrossing\\nAddition",\n"81": "Angles\\nSharp\\nJagged",\n"82": "Spiral\\nDrunkeness\\nCoil",\n"83": "Wavy\\nRipple\\nHair",\n"84": "Circle\\nRing",\n"85": "Round",\n"86": "Triangle",\n"87": "Star",\n"88": "Rectangle\\nSquare",\n"89": "Flat",\n"90": "Cube",\n"91": "Sphere",\n"92": "Pyramid",\n"93": "Cylinder",\n"94": "Cone",\n"95": "Hollow\\nHole\\nPierced",\n"96": "Big\\nHigh",\n"97": "Small\\nLow",\n"98": "Fat\\nLarge\\nLong",\n"99": "Thin - Fine\\nNarrow\\nShort",\n"100": "High\\nClimb\\nAbove",\n"101": "Low\\nDescend\\nBelow",\n"102": "Left\\nBeginning\\nBefore - Past",\n"103": "Right\\nEnd\\nAfter - Future",\n"104": "Turn\\nAround\\nCycle - Repetition",\n"105": "Use - Action\\nDo - Verbe\\nButton",\n"106": "Red",\n"107": "Orange",\n"108": "Yellow",\n"109": "Green",\n"110": "Blue",\n"111": "Purple",\n"112": "Pink",\n"113": "Brown",\n"114": "Black",\n"115": "Grey",\n"116": "White",\n"117": "Transparent\\nInvisible\\nGlass"\n}'

submission_code = '''
import os, math, json
import numpy as np
from sentence_transformers import SentenceTransformer

HINT = {int(k): v for k, v in json.loads(HINT_JSON).items()}
MEMBERS = [
    {"name": "bge", "dir": "./model_bge", "qp": "", "cp": ""},
    {"name": "e5",  "dir": "./model_e5",
     "qp": "Instruct: Retrieve the word matching the concept attributes.\\nQuery: ", "cp": ""},
]
_models = [(SentenceTransformer(mb["dir"]), mb) for mb in MEMBERS]

def hint_terms(hid):
    raw = HINT[hid].replace(" - ", "\\n").replace("-", "\\n")
    return [t.strip() for t in raw.split("\\n") if t.strip()]

def hints_to_query(hints):
    terms = []
    for h in hints:
        terms.extend(hint_terms(h))
    seen = set(); uniq = [t for t in terms if not (t.lower() in seen or seen.add(t.lower()))]
    return "A thing related to: " + ", ".join(uniq) + "."

def _znorm(a):
    return (a - a.mean(1, keepdims=True)) / (a.std(1, keepdims=True) + 1e-9)

def guess_words(hints, options):
    acc = None
    for m, mb in _models:
        q = m.encode([mb["qp"] + hints_to_query(hints)], convert_to_numpy=True, normalize_embeddings=True)
        c = m.encode([mb["cp"] + o for o in options], convert_to_numpy=True, normalize_embeddings=True)
        s = _znorm(q @ c.T)
        acc = s if acc is None else acc + s
    order = np.argsort(-acc[0])[:10]
    return [options[i] for i in order]
'''.replace("HINT_JSON", "r\"\"\"" + HINT_EMBED + "\"\"\"")

with open("submission_model.py", "w") as f:
    f.write(submission_code)
print("submission_model.py 작성 완료")

# 로컬 검증: submission_model.guess_words 로 held-out 재평가(노트북 함수와 일치해야 함)
import importlib, sys
sys.path.insert(0, ".")
import submission_model as sm
if TEST:
    chk = np.mean([total_score(sm.guess_words(e["hints"], e["options"]), e["label"]) for e in TEST])
    print(f"submission_model 검증 test_set: {chk:.4f}")


In [ ]:
import shutil, tempfile
# submission.zip = submission_model.py + 두 모델 디렉터리
with tempfile.TemporaryDirectory() as tmp:
    shutil.copy("submission_model.py", tmp)
    shutil.copytree("./model_bge", os.path.join(tmp, "model_bge"))
    shutil.copytree("./model_e5",  os.path.join(tmp, "model_e5"))
    shutil.make_archive("submission", "zip", tmp)
print("submission.zip 생성:", round(os.path.getsize("submission.zip")/1e6, 1), "MB")

# 파라미터 한도(각 50MB+ 항목 ≤ 1B) 점검
from safetensors import safe_open
for d in ["model_bge", "model_e5"]:
    tot = 0
    for st in glob.glob(f"{d}/**/*.safetensors", recursive=True):
        with safe_open(st, framework="pt") as f:
            tot += sum(f.get_slice(k).get_shape()[0] if False else np.prod(f.get_slice(k).get_shape()) for k in f.keys())
    print(f"  {d}: {int(tot):,} params  ({'OK ≤1B' if tot <= 1_000_000_000 else 'FAIL >1B'})")


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.zip']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)